In [0]:
spark.sql("USE workspace.steam")

# 1. the latest run of each layer
display(spark.sql("""
    SELECT layer, file_processed, status, rows_inserted, rows_updated,
           rows_skipped_unchanged, rows_quarantined,
           ROUND(unix_timestamp(end_time) - unix_timestamp(start_time)) AS seconds,
           start_time
    FROM (SELECT *, row_number() OVER (PARTITION BY layer ORDER BY start_time DESC) AS rn
          FROM pipeline_execution_logs
          WHERE layer IN ('Raw-to-Bronze', 'Bronze-to-Silver', 'Silver-to-Gold'))
    WHERE rn = 1
    ORDER BY start_time
"""))

# 2. size of every table and when it was last written
tables = ["steam.bronze_steam", "steam.silver_steam", "steam.silver_steam_history",
          "steam.silver_quarantine", "steam.schema_drift_log", "steam.pipeline_execution_logs",
          "gold.dim_date", "gold.dim_game", "gold.dim_publisher", "gold.dim_developer",
          "gold.fact_game_daily_snapshot", "gold.fact_price_change",
          "gold.fact_game_listing", "gold.fact_pipeline_run"]
rows = []
for t in tables:
    ts = "end_time" if t.endswith("pipeline_execution_logs") else "load_timestamp"
    r = spark.sql(f"SELECT COUNT(*) AS n, MAX({ts}) AS last_write FROM workspace.{t}").first()
    rows.append((t, r.n, r.last_write))
display(spark.createDataFrame(rows, "table string, rows bigint, last_write timestamp"))

layer,file_processed,status,rows_inserted,rows_updated,rows_skipped_unchanged,rows_quarantined,seconds,start_time
Raw-to-Bronze,/Volumes/workspace/default/steam/steamspy_2026-10-25.jsonl,Success,1234,0,81287,0,39,2026-10-25T06:58:01.843Z
Bronze-to-Silver,bronze_steam load_date=2026-10-25,Success,0,1228,0,0,22,2026-10-25T06:59:10.932Z
Silver-to-Gold,gold date=2026-10-25,Success,83621,1,null,null,74,2026-10-25T06:59:47.332Z


table,rows,last_write
steam.bronze_steam,119226,2026-10-25T06:58:28.212Z
steam.silver_steam,82400,2026-10-25T06:59:16.517Z
steam.silver_steam_history,118613,2026-10-25T06:59:27.517Z
steam.silver_quarantine,1,2026-10-10T06:34:19.980Z
steam.schema_drift_log,1,2026-10-10T06:33:46.252Z
steam.pipeline_execution_logs,99,2026-10-25T07:01:01.204Z
gold.dim_date,17,2026-10-25T06:59:49.025Z
gold.dim_game,82400,2026-10-25T06:59:56.916Z
gold.dim_publisher,44462,2026-10-25T07:00:05.746Z
gold.dim_developer,50309,2026-10-25T07:00:12.760Z
